# ColPali on the scanned PDF

`20260914-NDRRMC-IRR-RA12287.pdf` has no text layer (see `ingestion_benchmarks.ipynb`), so the text chunker and BM25 see nothing. **ColPali is not OCR**: it embeds each page *image* as many small vectors and scores a question against them (late interaction). It can find the right page, but it does not return text.

This notebook checks whether a small ColPali-family model (`vidore/colSmol-500M`, about 1 GB, fits the 6 GB GPU) retrieves the right page of the scanned PDF.

Needs `HF_TOKEN` in `backend/.env`. Relevance labels below come from pages I read by eye (1 and 5), so pages not listed may also be relevant; recall here is a floor.

In [1]:
import os
import sys
import time
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "corpus").is_dir())  # repo root, wherever the notebook lives
PDF = REPO / "corpus" / "20260914-NDRRMC-IRR-RA12287.pdf"

# Read HF_TOKEN from backend/.env without printing it.
from dotenv import load_dotenv
load_dotenv(REPO / "backend" / ".env")
assert os.getenv("HF_TOKEN"), "HF_TOKEN not found in backend/.env"

## 1. Render pages to images

In [2]:
import pymupdf
from PIL import Image

RENDER_DPI = 110  # the model resizes pages itself; higher DPI only costs memory


def render_pages(path, dpi=RENDER_DPI):
    """Return one PIL image per PDF page."""
    images = []
    with pymupdf.open(str(path)) as pdf:
        for page in pdf:
            pixmap = page.get_pixmap(dpi=dpi)
            images.append(Image.frombytes("RGB", (pixmap.width, pixmap.height), pixmap.samples))
    return images


page_images = render_pages(PDF)
print(len(page_images), "pages, first page size:", page_images[0].size)

17 pages, first page size: (910, 1287)


## 2. Load the model

In [3]:
import torch
from colpali_engine.models import ColIdefics3, ColIdefics3Processor

MODEL_ID = "vidore/colSmol-500M"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

model = ColIdefics3.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map=DEVICE).eval()
processor = ColIdefics3Processor.from_pretrained(MODEL_ID)
print("loaded on", DEVICE)

/home/kenneth28/my_projects/Flood-Gentic/backend/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/490 [00:00<?, ?it/s]

Loading weights:   1%|          | 4/490 [00:00<00:13, 35.64it/s]

Loading weights:  18%|█▊        | 89/490 [00:00<00:00, 487.68it/s]

Loading weights:  30%|██▉       | 145/490 [00:00<00:00, 517.40it/s]

...

Loading weights:  28%|██▊       | 127/448 [00:00<00:00, 1260.86it/s]

Loading weights:  57%|█████▋    | 254/448 [00:00<00:00, 1238.62it/s]

Loading weights:  85%|████████▌ | 383/448 [00:00<00:00, 1261.28it/s]

Loading weights: 100%|██████████| 448/448 [00:00<00:00, 1263.73it/s]

loaded on cuda


## 3. Embed the pages

In [4]:
PAGE_BATCH = 2  # lower to 1 if the GPU runs out of memory

page_embeddings = []
start = time.perf_counter()
for i in range(0, len(page_images), PAGE_BATCH):
    batch = processor.process_images(page_images[i : i + PAGE_BATCH]).to(model.device)
    with torch.no_grad():
        output = model(**batch)
    page_embeddings.extend(list(output.cpu().float()))  # one (patches, dim) tensor per page
seconds = time.perf_counter() - start

print(f"{len(page_embeddings)} pages in {seconds:.1f}s, {page_embeddings[0].shape[0]} vectors per page")
if DEVICE == "cuda":
    print("peak GPU MB:", round(torch.cuda.max_memory_allocated() / 1e6))

17 pages in 11.8s, 875 vectors per page
peak GPU MB: 1695


## 4. Retrieval check
Pages are 1-indexed as printed on the document ("Page 1 of 17"). Page 1 holds Rules I Sections 1-5 (title, purpose, policy, definitions); page 5 holds RDRRMC and local government rules and who can lift a State of Imminent Disaster.

In [5]:
QUESTIONS = [
    # (question, pages known to answer it)
    ("What is the title of this IRR?", {1}),
    ("What is the purpose of the implementing rules for RA 12287?", {1}),
    ("How does the IRR define anticipatory actions?", {1}),
    ("What is a cascading hazard?", {1}),
    ("Who may declare a State of Imminent Disaster in a locality?", {5}),
    ("How soon must the RDRRMC give a recommendation when an LGU requests a declaration?", {5}),
    ("Can BARMM adopt the mechanism for recommending a declaration?", {5}),
    ("Who has the authority to lift a state of imminent disaster?", {5}),
]

query_batch = processor.process_queries([q for q, _ in QUESTIONS]).to(model.device)
with torch.no_grad():
    query_embeddings = list(model(**query_batch).cpu().float())

# scores[q][p] = late-interaction score of question q against page p
scores = processor.score_multi_vector(query_embeddings, page_embeddings)

In [6]:
import pandas as pd

rows = []
for (question, answer_pages), question_scores in zip(QUESTIONS, scores):
    ranking = [int(i) + 1 for i in question_scores.argsort(descending=True)]  # 1-indexed pages
    first_hit = next(rank for rank, page in enumerate(ranking, 1) if page in answer_pages)
    rows.append({
        "question": question, "expected_page": sorted(answer_pages),
        "top3_pages": ranking[:3], "rank_of_first_hit": first_hit,
    })
results = pd.DataFrame(rows)
display(results)
print("recall@1:", (results.rank_of_first_hit <= 1).mean())
print("recall@3:", (results.rank_of_first_hit <= 3).mean())
print("MRR:", round((1 / results.rank_of_first_hit).mean(), 3))

,question,expected_page,top3_pages,rank_of_first_hit
0,What is the title of this IRR?,[1],"[3, 16, 2]",7
1,What is the purpose of the implementing rules ...,[1],"[1, 3, 4]",1
2,How does the IRR define anticipatory actions?,[1],"[13, 15, 6]",5
3,What is a cascading hazard?,[1],"[1, 2, 14]",1
4,Who may declare a State of Imminent Disaster i...,[5],"[5, 6, 15]",1
5,How soon must the RDRRMC give a recommendation...,[5],"[5, 6, 4]",1
6,Can BARMM adopt the mechanism for recommending...,[5],"[5, 4, 7]",1
7,Who has the authority to lift a state of immin...,[5],"[5, 4, 3]",1


recall@1: 0.75
recall@3: 0.75
MRR: 0.793


## Reading the results
- High recall@3 means ColPali can route a question to the right page of a scanned PDF without OCR.
- To use it in the RAG pipeline you still need a way to turn a retrieved page into text for the LLM (OCR the page, or send the page image to a vision model). That is a separate decision from this benchmark.
- Alternative for plain text: OCR the PDF with Tesseract (`/usr/bin/tesseract` is installed) and feed it through the normal chunker, which also keeps BM25 and metadata filters working.